In [ ]:
library(clusterProfiler)
library(org.Mm.eg.db)  # For mouse genes
library(dplyr)
library(enrichplot)
library(ggplot2)

# Common genes -> merge tables

In [ ]:
# 1. Read the kidney rescued-to-normal table
kidney_file <- "/data/projects/manuela/aging/aging_bulk/results_tables/kidney_rescued_to_normal.csv"
kidney_rescued <- read.csv(kidney_file, stringsAsFactors = FALSE)

# 2. Read the muscle rescued-to-normal table
muscle_file <- "/data/projects/manuela/aging/aging_bulk_muscle/results_tables/muscle_rescued_to_normal.csv"
muscle_rescued <- read.csv(muscle_file, stringsAsFactors = FALSE)

# 3. Decide which identifier to merge on (e.g., "X" or "Symbol")
# Here we use "X" (rownames saved as column) or "Symbol" if preferred
common_genes <- intersect(kidney_rescued$X, muscle_rescued$X)

# 4. Subset original tables for only the common genes
kidney_common <- kidney_rescued %>% filter(X %in% common_genes)
muscle_common <- muscle_rescued %>% filter(X %in% common_genes)

# 5. Merge the tables by gene ID
combined_table <- merge(kidney_common, muscle_common, by = "X", suffixes = c("_kidney", "_muscle"))

# 6. Optional: reorder columns (gene ID first)
combined_table <- combined_table %>% dplyr::select(X, everything())

#filter
# Columns to remove
cols_to_remove <- c("baseMean_kidney", "lfcSE_kidney", "stat_kidney", "pvalue_kidney",
                    "baseMean_muscle", "lfcSE_muscle", "stat_muscle", "pvalue_muscle")

# Remove unwanted columns
combined_table_clean <- combined_table %>% 
  dplyr::select(-all_of(cols_to_remove))

# Rename X to Gene_ID
combined_table_clean <- dplyr::rename(combined_table_clean, Gene_ID = X)
#change order
combined_table_clean <- combined_table_clean %>% dplyr::select(Gene_ID, Symbol_kidney, Symbol_muscle, everything())

head(combined_table_clean)

In [ ]:
library(ggplot2)
library(ggrepel)

# Define threshold for labeling outliers (example: abs difference > 1)
combined_table_clean$outlier <- (abs(combined_table_clean$log2FoldChange_kidney - combined_table_clean$log2FoldChange_muscle) > 1)|abs(combined_table_clean$log2FoldChange_muscle - combined_table_clean$log2FoldChange_kidney) < 1

ggplot(combined_table_clean, aes(x = log2FoldChange_kidney, y = log2FoldChange_muscle)) +
  geom_point(alpha = 0.5) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
  geom_text_repel(data = subset(combined_table_clean, outlier),
                  aes(label = Symbol_kidney),  # or use Symbol_muscle / Gene_ID
                  size = 3,
                  box.padding = 0.4,
                  point.padding = 0.3,
                  segment.color = 'grey50') +
  labs(title = "Log2 Fold Change Kidney vs Muscle with Outlier Gene Labels",
       x = "Kidney log2 Fold Change",
       y = "Muscle log2 Fold Change") +
  theme_minimal()

In [ ]:
# 7. Write to CSV
output_file <- "/data/projects/manuela/aging/aging_bulk/results_tables/common_rescued_genes_kidney_muscle.csv"
write.csv(combined_table_clean, output_file, row.names = FALSE, quote = FALSE)

# Enrichment Anaylsis

In [ ]:
# Extract gene IDs
genes_common <- combined_table_clean$Gene_ID

ego <- enrichGO(gene = genes_common ,
                OrgDb = org.Mm.eg.db,
                keyType = "ENSEMBL",
                ont = "BP",          # Biological Process
                pAdjustMethod = "BH",
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.2)

# View top results
head(ego)

In [ ]:
# genes_common is your vector of ENSEMBL IDs
entrez_ids <- mapIds(org.Mm.eg.db,
                     keys = genes_common,
                     column = "ENTREZID",
                     keytype = "ENSEMBL",
                     multiVals = "first")
entrez_ids <- na.omit(entrez_ids)  # remove any unmapped IDs

# Now run KEGG enrichment
ekegg <- enrichKEGG(gene = entrez_ids,
                    organism = 'mmu',
                    pvalueCutoff = 0.05)

# View results
head(ekegg)

In [ ]:
## Genes involved 
ego_readable <- setReadable(ego, OrgDb = org.Mm.eg.db, keyType = "ENSEMBL")
result_table <- as.data.frame(ego_readable)

head(result_table$geneID)

# Visualization

In [ ]:
# Dotplot for GO BP
dotplot(ego, showCategory=20) + ggtitle("GO BP Enrichment: Common Rescued Genes")
barplot(ego, showCategory=20) + ggtitle("GO BP Enrichment: Common Rescued Genes")

# Barplot for KEGG
barplot(ekegg, showCategory=15, title="KEGG Pathway Enrichment")